# C9-dimensionality-reduction — Practice p08 — Solution

In [1]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
n = 600
t = np.sort(rng.uniform(-1.5 * np.pi, 1.5 * np.pi, n))
y = rng.uniform(0.0, 6.0, n)
X3 = np.column_stack([np.sin(t), y, np.sign(t) * (np.cos(t) - 1.0)]) \
     + rng.normal(0.0, 0.03, (n, 3))
Xc = X3 - X3.mean(axis=0)
_, _, Vt = np.linalg.svd(Xc, full_matrices=False)
P2 = Xc @ Vt[:2].T
UV = np.column_stack([t, y])

def knn_preservation(A_orig, A_view, k):
    sq_orig = (A_orig * A_orig).sum(axis=1)
    D2_orig = np.maximum(sq_orig[:, None] + sq_orig[None, :] - 2.0 * (A_orig @ A_orig.T), 0.0)
    sq_view = (A_view * A_view).sum(axis=1)
    D2_view = np.maximum(sq_view[:, None] + sq_view[None, :] - 2.0 * (A_view @ A_view.T), 0.0)
    np.fill_diagonal(D2_orig, np.inf)
    np.fill_diagonal(D2_view, np.inf)
    nb_orig = np.argsort(D2_orig, axis=1)[:, :k]
    nb_view = np.argsort(D2_view, axis=1)[:, :k]
    fractions = np.empty(A_orig.shape[0], dtype=np.float64)
    for i in range(A_orig.shape[0]):
        fractions[i] = np.intersect1d(nb_orig[i], nb_view[i]).size / k
    return float(fractions.mean())

frac_pca = float(knn_preservation(X3, P2, 10))
frac_unroll = float(knn_preservation(X3, UV, 10))
local_winner = "unrolled" if frac_unroll > frac_pca else "pca"
print("local preservation, PCA / unrolled:", frac_pca, frac_unroll)
print("winner:", local_winner)

local preservation, PCA / unrolled: 0.5876666666666666 0.9321666666666666
winner: unrolled


The unrolled view wins the local-neighborhood comparison: it preserves $0.9321667$ of the $10$-neighbor memberships, versus $0.5876667$ for the two-dimensional principal-component view.

### Answer check

In [2]:
assert P2.shape == (600, 2)
assert UV.shape == (600, 2)
assert np.isclose(frac_pca, 0.5876666666666666, atol=1e-12, rtol=0)
assert np.isclose(frac_unroll, 0.9321666666666666, atol=1e-12, rtol=0)
assert 0.0 <= frac_pca <= 1.0 and 0.0 <= frac_unroll <= 1.0
assert local_winner == "unrolled"